In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
import numpy as np
from scipy.stats import truncnorm
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import image as img
%matplotlib notebook

In [2]:
if torch.cuda.is_available():
    print("GPU is available!")
else:
    print("GPU is not available.")

GPU is available!


双卷积模块，每一个编码器和解码器通过双卷积模块进行

In [3]:
class DoubleConv(nn.Module):
    """
    DoubleConv模块包含两个连续的卷积层，每个卷积层后接BatchNorm和ReLU激活函数。
    主要用于提取特征，同时保证特征尺寸不变
    """
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            # 第一个卷积层
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            # 第二个卷积层
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)

U-net网络，由若干次编码、解码构成，这里取4次

In [4]:
class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        """
        初始化UNet模型。
        :param in_channels: 输入通道数
        :param out_channels: 输出通道数
        """
        super(UNet, self).__init__()

        # 编码器部分（下采样路径）
        self.encoder1 = DoubleConv(in_channels, 32)  # 第一层编码器
        self.encoder2 = DoubleConv(32, 64)         # 第二层编码器
        self.encoder3 = DoubleConv(64, 128)       # 第三层编码器
        self.encoder4 = DoubleConv(128, 256)       # 第四层编码器
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)  # 最大池化层用于下采样

        # 瓶颈层
        self.bottleneck = DoubleConv(256, 512)  # 最底部特征提取层

        # 解码器部分（上采样路径）
        self.upconv4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)  # 上采样操作
        self.decoder4 = DoubleConv(512, 256)  # 解码器与跳跃连接结合特征

        self.upconv3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)  # 上采样操作
        self.decoder3 = DoubleConv(256, 128)  # 解码器与跳跃连接结合特征

        self.upconv2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)  # 上采样操作
        self.decoder2 = DoubleConv(128, 64)  # 解码器与跳跃连接结合特征

        self.upconv1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)  # 上采样操作
        self.decoder1 = DoubleConv(64, 32)  # 解码器与跳跃连接结合特征

        # 输出层，将特征图映射为分割结果
        self.final_conv = nn.Conv2d(32, out_channels, kernel_size=1)  # 输出卷积

    def forward(self, x):
        """
        定义UNet的前向传播。
        :param x: 输入图像
        :return: 分割结果张量
        """
        # 编码器路径：提取特征并逐步减少空间分辨率
        enc1 = self.encoder1(x)  # 第一层编码器
        enc2 = self.encoder2(self.pool(enc1))  # 第二层编码器
        enc3 = self.encoder3(self.pool(enc2))  # 第三层编码器
        enc4 = self.encoder4(self.pool(enc3))  # 第四层编码器

        # 瓶颈层，连接到解码器
        bottleneck = self.bottleneck(self.pool(enc4))

        # 解码器路径：逐步恢复空间分辨率，并结合编码器的跳跃连接
        dec4 = self.upconv4(bottleneck)  # 上采样第4层
        dec4 = torch.cat((enc4, dec4), dim=1)  # 跳跃连接结合编码器第4层
        dec4 = self.decoder4(dec4)  # 解码第4层

        dec3 = self.upconv3(dec4)  # 上采样第3层
        dec3 = torch.cat((enc3, dec3), dim=1)  # 跳跃连接结合编码器第3层
        dec3 = self.decoder3(dec3)  # 解码第3层

        dec2 = self.upconv2(dec3)  # 上采样第2层
        dec2 = torch.cat((enc2, dec2), dim=1)  # 跳跃连接结合编码器第2层
        dec2 = self.decoder2(dec2)  # 解码第2层

        dec1 = self.upconv1(dec2)  # 上采样第1层
        dec1 = torch.cat((enc1, dec1), dim=1)  # 跳跃连接结合编码器第1层
        dec1 = self.decoder1(dec1)  # 解码第1层

        # 最终输出层：生成分割结果
        return self.final_conv(dec1)

构建数据集

In [5]:
class CustomDataset(Dataset):
    def __init__(self, images, masks, transform=None):
        self.images = images
        self.masks = masks
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        mask = self.masks[idx]
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"]
        return image, mask

加载模型，定义损失函数、优化器

In [6]:
model = UNet(in_channels=1, out_channels=1).to('cuda') # 加载模型
criterion = nn.BCEWithLogitsLoss()  # 损失函数
optimizer = optim.Adam(model.parameters(), lr=1e-4) # 优化器

定义模型训练逻辑

In [7]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()  # 将模型设置为训练模式，启用 dropout 和 batch normalization 等特性
    epoch_loss = 0  # 初始化累计损失

    for images, masks in dataloader:  # 遍历数据加载器中的每个批次
        images, masks = images.to(device), masks.to(device)  # 将数据转移到指定设备（CPU 或 GPU）

        # Forward pass
        outputs = model(images)  # 前向传播：通过模型计算输出
        loss = criterion(outputs, masks)  # 根据输出和真实标签计算损失

        # Backward pass
        optimizer.zero_grad()  # 清零梯度，防止梯度累积
        loss.backward()  # 反向传播：计算梯度
        optimizer.step()  # 优化器更新模型参数

        epoch_loss += loss.item()  # 累加当前批次的损失值

    return epoch_loss / len(dataloader)  # 返回平均损失值


def validate(model, dataloader, criterion, device):
    model.eval()  # 将模型设置为评估模式，禁用 dropout 和 batch normalization 的随机特性
    val_loss = 0  # 初始化累计损失

    with torch.no_grad():  # 禁用梯度计算，加速和节省显存
        for images, masks in dataloader:  # 遍历验证集的每个批次
            images, masks = images.to(device), masks.to(device)  # 数据转移到设备
            outputs = model(images)  # 前向传播：通过模型计算输出
            loss = criterion(outputs, masks)  # 计算损失
            val_loss += loss.item()  # 累加当前批次的损失值

    return val_loss / len(dataloader)  # 返回平均损失值

产生训练数据

In [14]:
def atrous(image):
    image_nan_to_0 = np.float32(np.nan_to_num(image, nan=0.0))
    coe = img.a_trous(image_nan_to_0, level=4)
    hq = coe.data[1] + coe.data[2]
    # hq = img.box_car_smoothing(hq, kernel_size=7)
    hq[np.isnan(image)] = np.nan
    return hq


def unsharp_mask(image):
    image_nan_to_0 = np.float32(np.nan_to_num(image, nan=0.0))
    unsharp_mask_image = img.unsharp_mask(image_nan_to_0, kernel_size=21, high_freq_only=True)
    unsharp_mask_image[np.isnan(image)] = np.nan
    return unsharp_mask_image

def simulation(x, t, dx=120, dt=3, num_range=(100, 200), seed=None):
    if seed is not None:
        np.random.seed(seed)

    life_span = {'mean': 100, 'std': 50} # s
    brightness = {'mean': 24, 'std': 8}
    period = {'min': 60, 'max': 120} # s
    amplitude = {'mean': 15, 'std': 3} # km/s
    width = 7
    noise = {'mean': 7, 'std': 17}

    x_vect = np.arange(0, x, dx)
    t_vect = np.arange(0, t, dt)

    space_time = np.zeros((len(t_vect), len(x_vect)))
    mask = np.copy(space_time)

    num = int(np.random.uniform(num_range[0], num_range[-1]))

    for i in range(num):
        begin_time = np.random.uniform(0, t)
        begin_x = np.random.uniform(0, x)
        life = truncnorm.rvs(-2, 3, loc=life_span['mean'], scale=life_span['std'])
        phase = np.random.uniform(0, 2*np.pi)
        T = np.random.uniform(period['min'], period['max'])
        I = truncnorm.rvs(-3, 3, loc=brightness['mean'], scale=brightness['std'])
        speed = truncnorm.rvs(-3, 3, loc=amplitude['mean'], scale=amplitude['std'])
        A = 0.5*np.cos(np.random.uniform(0, 0.5*np.pi))*speed*T/np.pi

        t_index = (np.arange(begin_time, begin_time+life, dt)/dt).astype(int)
        t_index = np.clip(t_index, 0, len(t_vect)-1)
        c_index = np.floor((begin_x + A*np.sin(2*np.pi * t_index*dt/T + phase) + np.random.normal(0, A/5, len(t_index)))/dx).astype(int)
        c_index = np.clip(c_index, 0, len(x_vect)-1)

        for j, time in enumerate(t_index):
            c = c_index[j]
            mask[time, c] = 1
            x_index = (np.array(range(width)) - width//2 + c).astype(int)
            x_index = np.clip(x_index, 0, len(x_vect)-1)
            for index in x_index:
                # space_time[time, index] = I * np.exp(-(index-c)**2 / (2 * (width/10)**2))
                space_time[time, index] = I * np.cos(np.pi/(2*(width//2))*(index-c))

    space_time += truncnorm.rvs(-3, 3, loc=noise['mean'], scale=noise['std'], size=(len(t_vect), len(x_vect)))
    return atrous(space_time), mask

def normalization(data, norm_range=(0, 1)):
    data_min = np.min(data)
    data_max = np.max(data)
    return norm_range[0] + (norm_range[1] - norm_range[0]) * (data - data_min) / (data_max - data_min)

In [15]:
num_simulations = 1000
space_time_data = []
mask_data = []

for i in tqdm(range(num_simulations), desc="Simulating data", total=num_simulations):
    space_time, mask = simulation(256*120, 256*3, num_range=(50, 200), seed=i)
    space_time_data.append(normalization(space_time)[None, :, :])
    mask_data.append(mask[None, :, :])

space_time_data = np.array(space_time_data).astype(np.float32)
mask_data = np.array(mask_data).astype(np.float32)

np.save('./pytorch/space_time_data.npy', space_time_data)
np.save('./pytorch/mask_data.npy', mask_data)

# space_time_data = np.load('./pytorch/space_time_data.npy')
# mask_data = np.load('./pytorch/mask_data.npy')

# 划分训练集和测试集
from sklearn.model_selection import train_test_split
# 第一次划分：训练集 (80%) 和剩余部分 (20%)
train_images, temp_images, train_masks, temp_masks = train_test_split(space_time_data, mask_data, test_size=0.2, random_state=111)
# 第二次划分：验证集 (10%) 和测试集 (10%)
val_images, test_images, val_masks, test_masks = train_test_split(temp_images, temp_masks, test_size=0.5, random_state=111)

train_dataset = CustomDataset(train_images, train_masks)
test_dataset = CustomDataset(test_images, test_masks)
val_dataset = CustomDataset(val_images, val_masks)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

Simulating data:   0%|          | 0/1000 [00:00<?, ?it/s]

训练模型

In [16]:
num_epochs = 50
best_val_loss = float("inf")
train_losses = []
val_losses = []

for epoch in tqdm(range(num_epochs), desc="Epoch", total=num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, 'cuda')
    val_loss = validate(model, val_loader, criterion, 'cuda')

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # 保存最优模型
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "./pytorch/best_unet_model.pth")

fig, ax = plt.subplots()
ax.clear()
ax.plot(range(1, 51), train_losses, label='Train Loss', marker='o')
ax.plot(range(1, 51), val_losses, label='Val Loss', marker='o')
ax.legend()
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.grid()
fig.savefig("./fig/loss.png", dpi=300)

Epoch:   0%|          | 0/50 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

In [17]:
model.load_state_dict(torch.load("./pytorch/best_unet_model.pth", weights_only=True))
model.eval()

test_loss = validate(model, test_loader, criterion, 'cuda')
print(f"Test Loss: {test_loss:.4f}")

Test Loss: 0.1174


In [18]:
import matplotlib.pyplot as plt

model.eval()
with torch.no_grad():
    for images, masks in test_loader:
        images, masks = images.to('cuda'), masks.to('cuda')
        outputs = model(images)
        predictions = torch.sigmoid(outputs) > 0.5  # 二分类阈值
        break

plt.figure(figsize=(12, 6))
plt.subplot(1, 3, 1)
plt.title("Input Image")
plt.imshow(images[0].cpu().squeeze(), cmap="gray")

plt.subplot(1, 3, 2)
plt.title("Ground Truth")
plt.imshow(masks[0].cpu().squeeze(), cmap="gray")

plt.subplot(1, 3, 3)
plt.title("Prediction")
plt.imshow(predictions[0].cpu().squeeze(), cmap="gray")

plt.show()

<IPython.core.display.Javascript object>

对真实时空图测试

In [19]:
import os
from datetime import datetime
import re
import Sunlimb
from matplotlib.colors import Normalize, LogNorm

file_path = "E:/python/projects/alfven/data/corrected/10moving/"
files = [os.path.abspath(os.path.join(file_path, file)) for file in os.listdir(file_path) if file.endswith('.fits')]


def extract_datetime(file_name):
    match = re.search(r'\d{8}T\d{6}', file_name)
    if match:
        return datetime.strptime(match.group(), '%Y%m%dT%H%M%S')
    else:
        return datetime.min


files = sorted(files, key=extract_datetime)[267:523]

In [20]:
deg_range = (174.5, 177.05)
d_deg = 0.01
d_time = 3
deg_vect = np.arange(deg_range[0], deg_range[1] + d_deg / 2, d_deg)
time_vect = np.arange(0, len(files) * d_time + 0.5 * d_time, d_time) / 60  # unit: minute
r = 7
fig_st, ax_st = plt.subplots(figsize=(5, 5))

st_data = Sunlimb.space_time_plot(ax=ax_st, files=files, deg_range=deg_range, r=r, d_time=3, d_deg=0.01, process=atrous,
                                  cmap='sdoaia171',
                                  title='space_time_plot r=7Mm', xlabel='degree', ylabel='time(min)', norm=Normalize(),
                                  colorbar=True)
ax_st.set_aspect('auto')

<IPython.core.display.Javascript object>

Processing files:   0%|          | 0/256 [00:00<?, ?it/s]

In [21]:
flat_data1 = st_data.flatten()
fig, ax = plt.subplots(figsize=(7, 4))
_ = ax.hist(flat_data1, bins='auto', density=True, alpha=0.5, color='blue', label='hri intensity')
ax.grid()

<IPython.core.display.Javascript object>

In [22]:
test_st = normalization(st_data[None, :, :])
st_tensor = torch.tensor(test_st, dtype=torch.float32).unsqueeze(0).to('cuda')

model.eval()
with torch.no_grad():
    output = model(st_tensor)
    predictions = torch.sigmoid(output) > 0.5
output_image = predictions.squeeze().cpu().numpy()

note_data = np.copy(st_data)
note_data[output_image] = np.nan

In [23]:
fig, axs = plt.subplots(1, 3, figsize=(12, 4))
axs[0].imshow(st_data, cmap='sdoaia171', origin='lower', extent=[deg_vect[0], deg_vect[-1], time_vect[0], time_vect[-1]])
axs[1].imshow(output_image, cmap='gray', origin='lower', extent=[deg_vect[0], deg_vect[-1], time_vect[0], time_vect[-1]])
axs[2].imshow(note_data, cmap='sdoaia171', origin='lower', extent=[deg_vect[0], deg_vect[-1], time_vect[0], time_vect[-1]])
axs[0].set_aspect('auto')
axs[0].set_xlabel('Degree')
axs[0].set_ylabel('Time')
axs[1].set_aspect('auto')
axs[1].set_xlabel('Degree')
axs[1].set_ylabel('Time')
axs[2].set_aspect('auto')
axs[2].set_xlabel('Degree')
axs[2].set_ylabel('Time')

fig.savefig('./fig/label_st_data.png', dpi=300)

<IPython.core.display.Javascript object>